# CLEAR-HPV — ROI Overlay Demo (Single Slide)

This notebook generates a **level-0 ROI overlay** for a single slide using:
- a **CLAM checkpoint** to project encoder features into **h-space** and compute attention
- a **KMeans model** (e.g., `CLEAR-hpv_rawh`) to assign each ROI tile to a concept
- an overlay renderer that draws colored rectangles per tile

Requirements (typical): `openslide-python`, `Pillow`, `h5py`, `torch`, `joblib`, `numpy`, `matplotlib`.


## 0. Configuration (edit paths)

Edit the paths below to match your environment.


In [ ]:
from pathlib import Path

# --- Slide + data ---
SVS_ROOT   = Path("HNSCC_slides")
FEAT_DIR   = Path("/common/users/wq50/CLAM/features/HPV_UNI2_features/h5_files")

# Slide id (must have: FEAT_DIR/<SLIDE_ID>.h5 and a WSI under SVS_ROOT)
SLIDE_ID   = "TCGA-BA-6869-01Z-00-DX1.6e58648e-3309-47bb-b2c7-b71bcd9dc69 b_001"

# --- Models ---
# CLAM checkpoint for this split (used to get h + attention)
CLAM_WEIGHT = Path("/common/users/wq50/CLAM/results/HPV_CLAM_50_mb_s1/s_9_checkpoint.pt")

# cluster model to evaluate 
cluster_MODEL = Path("cluster_models/hpv_uni2_k10_rawh.joblib")

# --- Output ---
OUT_DIR = Path(f"./demo_overlay_{SLIDE_ID}"); OUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Params ---
EMBED_DIM = 1536
TILE_SIZE_L0 = 256
BATCH = 16_384

# ROI selection (choose one):
ROI_MODE = "relative"  # 'relative' | 'manual'
ROI_REL  = (0.50, 0.25, 0.10, 0.10)  # (cx_rel, cy_rel, w_rel, h_rel) in [0,1]
ROI_BOX_L0 = (10000, 12000, 16000, 18000)  # (x0,y0,x1,y1) at level-0 if ROI_MODE='manual'

# Overlay alpha (0..255)
ALPHA = 120


## 1. Imports + helpers


In [ ]:
import os, gc, re
import numpy as np
import h5py, joblib
import torch
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import openslide

SVS_EXTS = {".svs", ".tif", ".tiff", ".ndpi", ".mrxs"}

def find_wsi_by_stem(root: Path, stem: str) -> Path | None:
    for p in root.rglob("*"):
        if p.suffix.lower() in SVS_EXTS and p.stem == stem:
            return p
    return None

def iter_h5(h5_path: Path, batch=BATCH):
    with h5py.File(h5_path, "r") as f:
        X = f["features"]; C = f["coords"]
        N = X.shape[0]
        for off in range(0, N, batch):
            yield off, X[off:off+batch][:].astype(np.float32), C[off:off+batch][:].astype(np.int32)

def base_palette_colors(K: int):
    base_hex = [
        "#E41A1C","#377EB8","#031B2E","#984EA3","#FF7F00",
        "#FFD92F","#F781BF","#66C2A5","#A65628","#999999",
    ]
    base = np.array([mcolors.to_rgb(h) for h in base_hex], dtype=float)
    reps = int(np.ceil(K / len(base_hex)))
    return (np.tile(base, (reps, 1))[:K] * 255).astype(np.uint8)

def rel_box_to_abs_level0(rel_box, slide_wh_l0, align=TILE_SIZE_L0):
    cxr, cyr, wr, hr = rel_box
    W0, H0 = slide_wh_l0
    wr = float(np.clip(wr, 0.0, 1.0)); hr = float(np.clip(hr, 0.0, 1.0))
    cx = float(np.clip(cxr, 0.0, 1.0)) * W0
    cy = float(np.clip(cyr, 0.0, 1.0)) * H0
    w = max(align, int(round(wr * W0)))
    h = max(align, int(round(hr * H0)))
    x0 = int(round(cx - w/2)); y0 = int(round(cy - h/2))
    x1 = x0 + w; y1 = y0 + h
    x0 = max(0, min(x0, W0 - 1)); y0 = max(0, min(y0, H0 - 1))
    x1 = max(1, min(x1, W0));     y1 = max(1, min(y1, H0))
    if x1 - x0 < w: x0 = max(0, x1 - w)
    if y1 - y0 < h: y0 = max(0, y1 - h)
    x0 = (x0 // align) * align
    y0 = (y0 // align) * align
    x1 = min(W0, x0 + ((w // align) * align))
    y1 = min(H0, y0 + ((h // align) * align))
    if x1 <= x0: x1 = min(W0, x0 + align)
    if y1 <= y0: y1 = min(H0, y0 + align)
    return (x0, y0, x1, y1)

def filter_by_roi(coords_l0: np.ndarray, roi_box_l0):
    x0,y0,x1,y1 = roi_box_l0
    return ((coords_l0[:,0] >= x0) & (coords_l0[:,0] < x1) &
            (coords_l0[:,1] >= y0) & (coords_l0[:,1] < y1))

def render_roi_overlay_level0(svs_path: Path, roi_box_l0, coords_roi_l0, labels_roi, colors_uint8,
                              tile_size_l0=TILE_SIZE_L0, alpha=120):
    x0, y0, x1, y1 = roi_box_l0
    edge_w, edge_h = x1 - x0, y1 - y0

    slide = openslide.OpenSlide(str(svs_path))
    region = slide.read_region((x0, y0), 0, (edge_w, edge_h))
    slide.close()

    rgba = region.convert("RGBA")
    bg = Image.new("RGBA", rgba.size, (255,255,255,255))
    roi_rgb = Image.alpha_composite(bg, rgba).convert("RGB")

    overlay = Image.new("RGBA", (edge_w, edge_h), (0,0,0,0))
    draw = ImageDraw.Draw(overlay, "RGBA")

    w = tile_size_l0
    for (tx, ty), lab in zip(coords_roi_l0, labels_roi):
        x = int(tx - x0); y = int(ty - y0)
        r,g,b = map(int, colors_uint8[int(lab)])
        draw.rectangle([x, y, x+w, y+w], fill=(r, g, b, alpha), outline=None)

    roi_overlay_rgb = Image.alpha_composite(roi_rgb.convert("RGBA"), overlay).convert("RGB")
    return roi_rgb, roi_overlay_rgb


## 2. Load CLAM + project to h-space with attention

This cell assumes `models.model_clam.CLAM_MB` is importable from your repo.


In [ ]:
from models.model_clam import CLAM_MB

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

@torch.inference_mode()
def load_clam(weight_path: Path, device: str, embed_dim: int):
    m = CLAM_MB(gate=True, size_arg="small", n_classes=2, embed_dim=embed_dim)
    sd = torch.load(weight_path, map_location=device)
    m.load_state_dict(sd, strict=False)
    return m.to(device).eval()

@torch.inference_mode()
def project_h_and_attention(block_np: np.ndarray, clam: CLAM_MB, device: str):
    x = torch.from_numpy(block_np).to(device)
    A_raw, h = clam.attention_net(x)  # A_raw: (m,2)
    a = torch.softmax(A_raw[:,0], dim=0)
    a = a * (a.numel() / a.sum())     # mean≈1
    return h.cpu().numpy().astype(np.float32), a.cpu().numpy().astype(np.float32)

h5_path = FEAT_DIR / f"{SLIDE_ID}.h5"
wsi_path = find_wsi_by_stem(SVS_ROOT, SLIDE_ID)
assert h5_path.exists(), f"H5 not found: {h5_path}"
assert wsi_path is not None and wsi_path.exists(), f"WSI not found for: {SLIDE_ID}"

clam = load_clam(CLAM_WEIGHT, DEVICE, EMBED_DIM)
km = joblib.load(KMEANS_MODEL)
print("Loaded KMeans k=", getattr(km, "n_clusters", None))

# Stream through the slide features -> collect h, coords, attention
H_list, C_list, A_list = [], [], []
for _, X, C in iter_h5(h5_path, BATCH):
    H, a = project_h_and_attention(X, clam, DEVICE)
    H_list.append(H)
    C_list.append(C)
    A_list.append(a)
    del X, H
    gc.collect()

H_all  = np.concatenate(H_list, axis=0).astype(np.float32)
coords = np.concatenate(C_list, axis=0).astype(np.int32)
attn   = np.concatenate(A_list, axis=0).astype(np.float32)
print("Tiles:", H_all.shape[0], "  h-dim:", H_all.shape[1])


## 3. Choose ROI, assign concepts, and render overlay


In [ ]:
# Get slide dimensions (level-0)
slide = openslide.OpenSlide(str(wsi_path))
W0, H0 = slide.dimensions
slide.close()
print("Slide dims (L0):", (W0, H0))

# ROI box at level-0
if ROI_MODE == "relative":
    roi_box = rel_box_to_abs_level0(ROI_REL, (W0, H0), align=TILE_SIZE_L0)
elif ROI_MODE == "manual":
    roi_box = ROI_BOX_L0
else:
    raise ValueError("ROI_MODE must be 'relative' or 'manual' in this demo notebook")

in_roi = filter_by_roi(coords, roi_box)
assert np.any(in_roi), "No tiles in ROI. Adjust ROI parameters."

# Assign clusters for ROI tiles using km.predict on h-space
H_roi = H_all[in_roi]
coords_roi = coords[in_roi]
labels_roi = km.predict(H_roi).astype(np.int32)

counts = np.bincount(labels_roi, minlength=km.n_clusters)
fracs = counts / max(1, counts.sum())
print("ROI box (L0):", roi_box)
print("Counts:", counts.tolist())
print("Fracs:", np.round(fracs, 3).tolist())

# Render overlay at level-0 ROI
colors = base_palette_colors(km.n_clusters)
roi_orig, roi_overlay = render_roi_overlay_level0(
    svs_path=wsi_path,
    roi_box_l0=roi_box,
    coords_roi_l0=coords_roi,
    labels_roi=labels_roi,
    colors_uint8=colors,
    tile_size_l0=TILE_SIZE_L0,
    alpha=ALPHA,
)

orig_path = OUT_DIR / f"{SLIDE_ID}_ROI_original.png"
ov_path   = OUT_DIR / f"{SLIDE_ID}_ROI_overlay.png"
roi_orig.save(orig_path)
roi_overlay.save(ov_path)
print("Saved:", orig_path)
print("Saved:", ov_path)

# Display
plt.figure(figsize=(10, 10))
plt.imshow(roi_overlay)
plt.axis('off')
plt.title("ROI Overlay (level-0)")
plt.show()


## 4. Optional: Save a small legend image


In [ ]:
from PIL import ImageFont

def save_legend(colors_uint8, out_path: Path, title="Concept Legend"):
    K = len(colors_uint8)
    row_h = 28
    w = 260
    h = 40 + K * row_h
    img = Image.new("RGB", (w, h), (255,255,255))
    draw = ImageDraw.Draw(img)
    try:
        font = ImageFont.load_default()
    except Exception:
        font = None
    draw.text((10, 10), title, fill=(0,0,0), font=font)
    for k, col in enumerate(colors_uint8):
        y = 40 + k * row_h
        r,g,b = map(int, col)
        draw.rectangle([10, y, 34, y+18], fill=(r,g,b), outline=(0,0,0))
        draw.text((45, y), f"C{k}", fill=(0,0,0), font=font)
    img.save(out_path)
    return out_path

legend_path = OUT_DIR / f"{SLIDE_ID}_legend.png"
save_legend(colors, legend_path)
print("Saved legend:", legend_path)

plt.figure(figsize=(3, 6))
plt.imshow(Image.open(legend_path))
plt.axis('off')
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# ---- input ----
fracs = np.array(
    [0.268, 0.002, 0.133, 0.058, 0.0, 0.513, 0.0, 0.0, 0.003, 0.024],
    dtype=float
)

# ---- palette (same as overlay) ----
def base_palette_colors(K: int):
    base_hex = [
        "#E41A1C","#377EB8","#031B2E","#984EA3","#FF7F00",
        "#FFD92F","#F781BF","#66C2A5","#A65628","#999999",
    ]
    base = np.array([mcolors.to_rgb(h) for h in base_hex], dtype=float)
    reps = int(np.ceil(K / len(base_hex)))
    return np.tile(base, (reps, 1))[:K]

# ---- plotting (DISPLAY ONLY) ----
def plot_cluster_fractions(fracs, orientation="v", sort=False, show_values=False):
    fracs = np.asarray(fracs, float)
    K = len(fracs)
    colors = base_palette_colors(K)

    idx = np.arange(K)
    labels = [f"C{i}" for i in idx]

    if sort:
        order = np.argsort(fracs)[::-1]
        fracs = fracs[order]
        colors = colors[order]
        labels = [labels[i] for i in order]

    if orientation == "v":
        fig, ax = plt.subplots(figsize=(7, 2.2), dpi=300)
        bars = ax.bar(np.arange(K), fracs, color=colors, edgecolor="black", width=0.8)

        ax.set_xticks(np.arange(K))
        ax.set_xticklabels(labels, fontsize=20)
        ax.set_ylabel("Fraction", fontsize=15)
        ax.set_ylim(0, max(0.01, fracs.max()) * 1.15)
        ax.set_yticks([0, 0.25, 0.5])
        ax.tick_params(axis="y", labelsize=20)

        # bold key concepts
        for i, tick in enumerate(ax.get_xticklabels()):
            if labels[i] in {"C5", "C7"}:
                tick.set_fontweight("bold")

    else:
        fig, ax = plt.subplots(figsize=(8.0, 2.2), dpi=300)
        y = np.arange(K)
        bars = ax.barh(y, fracs, color=colors, edgecolor="black", height=0.8)

        ax.set_yticks(y)
        ax.set_yticklabels(labels, fontsize=12)
        ax.set_xlabel("Fraction", fontsize=12)
        ax.set_xlim(0, max(0.01, fracs.max()) * 1.15)

    # grid and spines
    ax.grid(axis="y" if orientation == "v" else "x", linewidth=0.4, alpha=0.4)
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)
    ax.spines["left"].set_linewidth(0.6)
    ax.spines["bottom"].set_linewidth(0.6)

    # optional value labels
    if show_values:
        for b, p in zip(bars, fracs):
            if p > 0:
                if orientation == "v":
                    ax.text(
                        b.get_x() + b.get_width() / 2,
                        b.get_height() + 0.01 * fracs.max(),
                        f"{p:.3f}",
                        ha="center",
                        va="bottom",
                        fontsize=14,
                    )
                else:
                    ax.text(
                        b.get_width() + 0.01 * fracs.max(),
                        b.get_y() + b.get_height() / 2,
                        f"{p:.3f}",
                        ha="left",
                        va="center",
                        fontsize=10,
                    )

    # emphasize C5 and C7 bars
    for i, lab in enumerate(labels):
        if lab in {"C5", "C7"}:
            bars[i].set_linewidth(2.5)

    fig.tight_layout(pad=0.5)
    plt.show()

# ---- run ----
plot_cluster_fractions(fracs, orientation="v", sort=False, show_values=False)
